# OASIS - Offline Adaptive Student & Instructor System

In [1]:
# 1. Install zstd (Required for Ollama extraction) and the Ollama Engine
print("Installing dependencies...")
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pip install -q -U gradio ollama transformers > /dev/null 2>&1

Installing dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
# 1. Install necessary Python packages
!pip install -q streamlit ollama langchain numpy scipy opencv-python-headless matplotlib
!pip install PyMuPDF
# 2. Install Ollama to the Linux environment
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start the Ollama server in the background
import subprocess
import time

print("Starting Ollama server...")
# In Kaggle, we route the output to avoid cluttering the cell
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)  # Give the server a few seconds to start

# 4. Pull the Gemma 4 26b model
print("Pulling Gemma 4:26b model...")
!ollama pull gemma4:26b
# Pull the Gemma 4 Edge model (e4b)
print("Pulling Gemma 4:e4b model... (This takes a minute or two)")
!ollama pull gemma4:e4b
print("Model pulled successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 67.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 108.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 68.5 MB/s eta 0:00:00:00:0100:01
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                                                  18.0%###                 79.3%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
Pulling Gemma 4:26b model...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling m

In [3]:
!pip install -q langchain langchain-community sentence-transformers faiss-cpu
!pip install langchain-core

import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Create the Mock 15-Year GATE JSON Database
db_data = [
    {
        "question_number": 11,
        "subject": "Engineering Mathematics",
        "topic": "Matrices",
        "question_text": "Consider the matrix A below. For which combinations...",
        "options": {
            "A": "Only (i), (iii), and (iv)",
            "B": "Only (iv)",
            "C": "Only (ii)",
            "D": "Only (i) and (iii)"
        },
        "has_circuit_or_graph": False,
        "detailed_solution": "The given matrix $A$ is an upper triangular matrix:\n$A = \\begin{pmatrix} 2 & 3 & 4 & 5 \\\\ 0 & 6 & 7 & 8 \\\\ 0 & 0 & \\alpha & \\beta \\\\ 0 & 0 & 0 & \\gamma \\end{pmatrix}$\nFor an upper triangular matrix, the rank is equal to the number of non-zero diagonal elements. The diagonal elements are $2, 6, \\alpha,$ and $\\gamma$. Since $2$ and $6$ are non-zero, the rank of $A$ will be at least 3 if at least one of the remaining diagonal elements, $\\alpha$ or $\\gamma$, is non-zero. \n\nLet's evaluate the given combinations:\n(i) $\\alpha = 0$ and $\\beta = \\gamma \\neq 0$: Here, $\\gamma \\neq 0$, so the diagonal elements are $2, 6, 0, \\gamma \\neq 0$. The rank is 3. (Satisfies condition)\n(ii) $\\alpha = \\beta = \\gamma = 0$: Here, $\\alpha = 0$ and $\\gamma = 0$. The diagonal elements are $2, 6, 0, 0$. The rank is 2. (Does not satisfy condition)\n(iii) $\\beta = \\gamma = 0$ and $\\alpha \\neq 0$: Here, $\\alpha \\neq 0$ and $\\gamma = 0$. The diagonal elements are $2, 6, \\alpha \\neq 0, 0$. The rank is 3. (Satisfies condition)\n(iv) $\\alpha = \\beta = \\unaligned \\gamma \\neq 0$: Here, $\\alpha \\neq 0$ and $\\gamma \\neq 0$. The diagonal elements are $2, 6, \\alpha \\neq 0, \\gamma \\neq 0$. The rank is 4. (Satisfies condition)\n\nThus, the combinations that result in a rank of at least 3 are (i), (iii), and (iv). This corresponds to option (A).",
        "predicted_answer": [
            "A"
        ],
        "official_answer": [
            "A"
        ],
        "is_correct": True,
        "year": 2025,
        "image_path": None
    },
    {
        "question_number": 12,
        "subject": "Engineering Mathematics",
        "topic": "Infinite Series",
        "question_text": "Consider the following series: (i) (ii) (iii)",
        "options": {
            "A": "Only (ii) converges",
            "B": "Only (ii) and (iii) converge",
            "C": "Only (iii) converges",
            "D": "All three converge"
        },
        "has_circuit_or_graph": False,
        "detailed_solution": "1. Series (i) is $\\sum_{n=1}^{\\infty} \\frac{1}{\\sqrt{n}}$, which is a p-series with $p = 1/2$. Since $p \\le 1$, the series diverges. 2. Series (ii) is $\\sum_{n=1}^{\\infty} \\frac{1}{n(n+1)}$. Using partial fractions, $\\frac{1}{n(n+1)} = \\frac{1}{n} - \\frac{1}{n+1}$. This is a telescoping series that converges to 1. 3. Series (iii) is $\\sum_{n=1}^{\\infty} \\frac{1}{n!}$. Using the ratio test, $\\lim_{n \\to \\infty} |\\frac{a_{n+1}}{a_n}| = \\lim_{n \\to \\infty} \\frac{n!}{(n+1)!} = \\lim_{n \\to \\infty} \\frac{1}{n+1} = 0$. Since $0 < 1$, the series converges. Therefore, only (ii) and (iii) converge.",
        "predicted_answer": [
            "B"
        ],
        "official_answer": [
            "B"
        ],
        "is_correct": True,
        "year": 2025,
        "image_path": None
    },
    {
        "question_number": 13,
        "subject": "General Aptitude",
        "topic": "Numerical Ability",
        "question_text": "A pot contains two red and two blue balls.",
        "options": {
            "A": "2/3",
            "B": "1/3",
            "C": "1/2",
            "D": "1"
        },
        "has_circuit_or_graph": False,
        "detailed_solution": "Total number of balls = 2 red + 2 blue = 4 balls. We are drawing 2 balls without replacement. The total number of ways to choose 2 balls from 4 is $C(4, 2) = \\frac{4 \\times 3}{2 \\times 1} = 6$. For the two balls to have different colors, one must be red and one must be blue. The number of ways to choose 1 red ball from 2 is $C(2, 1) = 2$, and the number of ways to choose 1 blue ball from 2 is $C(2, 1) = 2$. Thus, the number of favorable outcomes is $2 \\times 2 = 4$. The probability is $\\frac{4}{6} = \\frac{2}{3}$.",
        "predicted_answer": [
            "A"
        ],
        "official_answer": [
            "A"
        ],
        "is_correct": True,
        "year": 2025,
        "image_path": None
    },
    {
        "question_number": 15,
        "subject": "Communication Systems",
        "topic": "Channel Capacity",
        "question_text": "AWGN channel capacity dependence on bandwidth W",
        "options": {
            "A": "Decreasing curve",
            "B": "Linear curve",
            "C": "Convex curve",
            "D": "Concave curve"
        },
        "has_circuit_or_graph": True,
        "detailed_solution": "The capacity $C$ of an AWGN channel with bandwidth $W$, signal power $P_{av}$, and noise power spectral density $N_0/2$ is given by the Shannon-Hartley theorem: $C = W \\log_2(1 + \\frac{P_{av}}{N_0 W})$. Let $k = \\frac{P_{av}}{N_0}$. Then $C(W) = W \\log_2(1 + \\frac{k}{W})$. \n1. As $W \\to 0$, $C(W) \\to 0$. \n2. As $W \\to \\infty$, $C(W) \\to \\frac{k}{\\ln 2} = \\frac{P_{av}}{N_0 \\ln 2}$, which is a constant value. \n3. To find the curvature, we look at the second derivative. Let $f(W) = W \\ln(1 + k/W)$. The second derivative $f''(W) = \\frac{-k^2}{W(W+k)^2}$, which is always negative for $W > 0$. \nSince the second derivative is negative, the function is concave (downward curvature). The plot must start at the origin, increase, and approach a horizontal asymptote. This matches option (D).",
        "predicted_answer": [
            "D"
        ],
        "official_answer": [
            "D"
        ],
        "is_correct": True,
        "year": 2025,
        "image_path": "/kaggle/working/images/Que_2025_Page14.jpg"
    },
    {
        "question_number": 18,
        "subject": "Signals and Systems",
        "topic": "Fourier Transform",
        "question_text": "Consider a continuous-time, real-valued signal f(t) whose Fourier transform",
        "options": {
            "A": "$|F(\\omega)| \\le \\int_{-\\infty}^{\\infty} |f(t)| dt$",
            "B": "$|F(\\omega)| > \\int_{-\\infty}^{\\infty} |f(t)| dt$",
            "C": "$|F(\\omega)| \\le \\int_{-\\infty}^{\\infty} f(t) dt$",
            "D": "$|F(\\omega)| \\ge \\int_{-\\infty}^{\\infty} f(t) dt$"
        },
        "has_circuit_or_graph": False,
        "detailed_solution": "The Fourier transform of a signal $f(t)$ is defined as $F(\\omega) = \\int_{-\\infty}^{\\infty} f(t) e^{-j \\omega t} dt$. To find the upper bound of the magnitude $|F(\\omega)|$, we use the property of integrals: $|\\int g(t) dt| \\le \\int |g(t)| dt$. Applying this to the Fourier transform formula, we get $|F(\\omega)| = |\\int_{-\\infty}^{\\infty} f(t) e^{-j \\omega t} dt| \\le \\int_{-\\infty}^{\\infty} |f(t) e^{-j \\omega t}| dt$. Since the magnitude of the complex exponential $|e^{-j \\omega t}|$ is always 1 for any real $\\omega$ and $t$, the inequality simplifies to $|F(\\omega)| \\le \\int_{-\\infty}^{\\infty} |f(t)| \\cdot 1 dt$, which is $|F(\\omega)| \\le \\int_{-\\infty}^{\\infty} |f(t)| dt$. This matches option (A).",
        "predicted_answer": [
            "A"
        ],
        "official_answer": [
            "A"
        ],
        "is_correct": True,
        "year": 2025,
        "image_path": None
    },
  {
    "question_number": 20,
    "subject": "Networks, Signals and Systems",
    "topic": "Network Solution Methods",
    "question_text": "Let i_C, i_L, and i_R be the currents flowing through",
    "options": {
      "A": "i_C = 0.25∠180° A, i_L = 0.1∠0° A, i_R = 0.2∠90° A",
      "B": "i_C = 4∠180° A, i_L = 10∠0° A, i_R = 5∠90° A",
      "C": "i_C = 0.25∠270° A, i_L = 0.1∠90° A, i_R = 0.2∠90° A",
      "D": "i_C = 4∠90° A, i_L = 10∠270° A, i_R = 5∠0° A"
    },
    "has_circuit_or_graph": True,
    "detailed_solution": "The circuit shows a capacitor, inductor, and resistor in parallel with an AC voltage source V = 1∠90° V. The admittances are Y_C = j0.25 S, Y_L = -j0.1 S, and Y_R = 0.2 S. Using Ohm's Law for admittances (I = V * Y): For the capacitor, i_C = (1∠90°) * (0.25∠90°) = 0.25∠180° A. For the inductor, i_L = (1∠90°) * (0.1∠-90°) = 0.1∠0° A. For the resistor, i_R = (1∠90°) * (0.2) = 0.2∠90° A. Option A matches these results perfectly.",
    "predicted_answer": ["A"],
    "official_answer": ["A"],
    "is_correct": True,
    "year": 2025,
    "image_path": "./images/Que_2025_Q20.jpg"
  },
  {
    "question_number": 21,
    "subject": "Analog Circuits",
    "topic": "BJT Amplifiers",
    "question_text": "A simplified small-signal equivalent circuit of a BJT-based amplifier",
    "options": {
      "A": "-\\beta R_L / (R_S + r_\\pi)",
      "B": "+\\beta R_L / R_S",
      "C": "-\\beta R_L / R_S",
      "D": "+\\beta R_L / (R_S + r_\\pi)"
    },
    "has_circuit_or_graph": True,
    "detailed_solution": "From the input loop, applying KVL: v_s = i_b(R_S + r_\\pi), which gives i_b = v_s / (R_S + r_\\pi). At the output, the dependent current source \\beta i_b flows downwards through R_L, meaning the current enters the ground node. Therefore, the output voltage is v_o = -(\\beta i_b) * R_L. Substituting i_b into this equation yields v_o = -\\beta * (v_s / (R_S + r_\\pi)) * R_L. Rearranging for voltage gain (v_o / v_s) gives -\\beta R_L / (R_S + r_\\pi).",
    "predicted_answer": ["A"],
    "official_answer": ["A"],
    "is_correct": True,
    "year": 2025,
    "image_path": "./images/Que_2025_Q21.jpg"
  },
  {
    "question_number": 22,
    "subject": "Analog Circuits",
    "topic": "BJT Biasing",
    "question_text": "The ideal BJT in the circuit given below is biased",
    "options": {
      "A": "4.95",
      "B": "3.03",
      "C": "1.92",
      "D": "3.73"
    },
    "has_circuit_or_graph": True,
    "detailed_solution": "This is a collector-to-base bias circuit. Given I_B = 10 \\mu A and \\beta = 100. The collector current is I_C = \\beta * I_B = 100 * 10\\mu A = 1 mA. The emitter current is I_E = I_C + I_B = 1.01 mA. The current flowing through the top 5k\\Omega resistor is the total current (I_C + I_B = 1.01 mA). The voltage at the collector node V_C = V_CC - (I_C + I_B)*5k\\Omega = 10 - (1.01mA * 5k\\Omega) = 10 - 5.05 = 4.95 V. The voltage at the emitter node V_E = I_E * 3k\\Omega = 1.01mA * 3k\\Omega = 3.03 V. The collector-emitter voltage V_CE = V_C - V_E = 4.95 - 3.03 = 1.92 V.",
    "predicted_answer": ["C"],
    "official_answer": ["C"],
    "is_correct": True,
    "year": 2025,
    "image_path": "./images/Que_2025_Q22.jpg"
  },
  {
    "question_number": 23,
    "subject": "Digital Circuits",
    "topic": "Logic Gates",
    "question_text": "A 3-input majority logic gate has inputs X, Y, and Z.",
    "options": {
      "A": "XY + YZ + ZX",
      "B": "X \\oplus Y \\oplus Z",
      "C": "X + Y + Z",
      "D": "XYZ"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "A majority logic gate outputs logic '1' if the majority of its inputs are '1'. For a 3-input gate (X, Y, Z), it needs at least two inputs to be '1'. The truth table minterms that satisfy this are when inputs are 011, 101, 110, and 111 (minterms 3, 5, 6, 7). Writing the Sum of Products (SOP) expression: F = X'YZ + XY'Z + XYZ' + XYZ. Simplifying this expression using Boolean algebra yields F = XY + YZ + ZX. This represents the standard carry-out function of a full adder, which is a majority gate.",
    "predicted_answer": ["A"],
    "official_answer": ["A"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
 {
    "question_number": 25,
    "subject": "Engineering Mathematics",
    "topic": "Calculus",
    "question_text": "Consider the function f: R -> R, defined as f(x)",
    "options": {
      "A": "f has no global maximizer",
      "B": "f has no global minimizer",
      "C": "x = -1 is a local minimizer of f",
      "D": "x = 2 is a local maximizer of f"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "Given f(x) = 2x^3 - 3x^2 - 12x + 1. Find the first derivative: f'(x) = 6x^2 - 6x - 12 = 6(x^2 - x - 2) = 6(x-2)(x+1). Setting f'(x) = 0 gives critical points at x = 2 and x = -1. Find the second derivative: f''(x) = 12x - 6. At x = 2, f''(2) = 18 > 0 (local minimum, making D false). At x = -1, f''(-1) = -18 < 0 (local maximum, making C false). Since it is a cubic polynomial with a positive leading coefficient, the function goes to infinity as x approaches infinity, and negative infinity as x approaches negative infinity. Thus, it has no global maximum or minimum.",
    "predicted_answer": ["A", "B"],
    "official_answer": ["A", "B"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 26,
    "subject": "Control Systems",
    "topic": "Root Locus",
    "question_text": "Consider the unity-negative-feedback system shown in Figure (i) below",
    "options": {
      "A": "K = 5",
      "B": "K = 1/5",
      "C": "For no positive value of K",
      "D": "For all positive values of K"
    },
    "has_circuit_or_graph": True,
    "detailed_solution": "From the root locus plot, the open-loop poles (X) are at s = -1 and s = -2. The open-loop zeros (O) are at s = -3 and s = -4. The open-loop transfer function is L(s) = K(s+3)(s+4) / ((s+1)(s+2)). For a closed-loop pole to exist at s = -1 + j1, it must satisfy the magnitude condition |L(s)| = 1. Substitute s = -1 + j1 into the equation: K = |(-1+j1)+1||(-1+j1)+2| / |(-1+j1)+3||(-1+j1)+4|. Calculating magnitudes: K = (1 * \\sqrt{2}) / (\\sqrt{5} * \\sqrt{10}) = \\sqrt{2} / \\sqrt{50} = \\sqrt{1/25} = 1/5.",
    "predicted_answer": ["B"],
    "official_answer": ["B"],
    "is_correct": True,
    "year": 2025,
    "image_path": "./images/Que_2025_Q26.jpg"
  },
  {
    "question_number": 27,
    "subject": "Signals and Systems",
    "topic": "Z-Transform",
    "question_text": "Let x[n] be a discrete-time signal whose Z-transform is X(z).",
    "options": {
      "A": "The discrete-time Fourier transform (DTFT) of x[n] always exists",
      "B": "The region of convergence (RoC) of X(z) contains neither poles nor zeros",
      "C": "The discrete-time Fourier transform (DTFT) exists if the region of convergence (RoC) contains the unit circle",
      "D": "If x[n] = \\alpha \\delta[n], where \\delta[n] is the unit impulse and \\alpha is a scalar, then the region of convergence (RoC) is the entire Z-plane"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "Statement C is true because the DTFT is simply the Z-transform evaluated on the unit circle (z = e^{j\\omega}); if the RoC includes the unit circle, the DTFT converges and exists. Statement D is true because the Z-transform of an impulse \\delta[n] is 1, meaning X(z) = \\alpha. Since there are no 'z' terms in the denominator, there are no poles, making the RoC the entire Z-plane. Statement A is false (DTFT doesn't exist for growing exponentials) and B is false (RoC cannot contain poles, but it CAN contain zeros).",
    "predicted_answer": ["C", "D"],
    "official_answer": ["C", "D"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 28,
    "subject": "Communications",
    "topic": "Amplitude Modulation",
    "question_text": "Consider a message signal m(t) which is bandlimited to [-W, W]",
    "options": {
      "A": "(i) is TRUE",
      "B": "(i) is FALSE",
      "C": "(ii) is TRUE",
      "D": "(ii) is FALSE"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "For statement (i) regarding DSB-SC: The envelope is |A_c m(t)|. If m(t) > 0 for all t, the envelope is perfectly proportional to m(t), meaning an envelope detector will perfectly recover the signal. Thus, (i) is TRUE. For statement (ii) regarding standard AM: The envelope is A_c|1 + \\mu m(t)|. An envelope detector works as long as (1 + \\mu m(t)) > 0, which prevents phase reversal. It does NOT strictly require m(t) > 0, only that the modulation index limits are respected. Thus, stating it works 'only if m(t) > 0' makes (ii) FALSE.",
    "predicted_answer": ["A", "D"],
    "official_answer": ["A", "D"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 29,
    "subject": "Analog Circuits",
    "topic": "Operational Amplifiers",
    "question_text": "Which of the following statements is/are TRUE with respect to an ideal opamp?",
    "options": {
      "A": "It has an infinite input resistance",
      "B": "It has an infinite output resistance",
      "C": "It has an infinite open-loop differential gain",
      "D": "It has an infinite open-loop common-mode gain"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "An ideal operational amplifier is characterized by an infinite input impedance/resistance (so it draws zero current at the input terminals), zero output resistance (so it acts as a perfect voltage source), infinite open-loop differential voltage gain, and zero common-mode gain (meaning infinite Common Mode Rejection Ratio). Therefore, statements A and C are the correct characteristics.",
    "predicted_answer": ["A", "C"],
    "official_answer": ["A", "C"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 30,
    "subject": "Analog Circuits",
    "topic": "MOSFET Amplifiers",
    "question_text": "Which of the following statements is/are TRUE with respect to ideal MOSFET-based DC-coupled single-stage amplifiers",
    "options": {
      "A": "The common-gate amplifier has an infinite input resistance",
      "B": "The common-source amplifier has an infinite input resistance",
      "C": "The input and output voltages of the common-source amplifier are in phase",
      "D": "The input and output voltages of the common-drain amplifier are in phase"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "For an ideal MOSFET at low frequencies/DC, the gate draws zero current, meaning input resistance at the gate is infinite. A Common-Source (CS) amplifier takes input at the gate, so B is true. A Common-Gate (CG) amplifier takes input at the source, which has a finite resistance (1/g_m), so A is false. The CS amplifier is an inverting amplifier (180 degrees out of phase), so C is false. The Common-Drain (CD) amplifier, also known as a source follower, has a voltage gain of approx 1 and is non-inverting (in phase), so D is true.",
    "predicted_answer": ["B", "D"],
    "official_answer": ["B", "D"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 31,
    "subject": "Electronic Devices",
    "topic": "Semiconductor Physics",
    "question_text": "Which of the following can be used as an n-type dopant",
    "options": {
      "A": "Arsenic",
      "B": "Boron",
      "C": "Gallium",
      "D": "Phosphorous"
    },
    "has_circuit_or_graph": False,
    "detailed_solution": "Silicon is a Group 14 element with 4 valence electrons. To create an n-type semiconductor (donor), a pentavalent element (Group 15, with 5 valence electrons) must be used so that one extra electron is free to conduct. Group 15 elements include Nitrogen, Phosphorous (P), Arsenic (As), Antimony, and Bismuth. Boron and Gallium are Group 13 elements (trivalent) and act as p-type acceptors.",
    "predicted_answer": ["A", "D"],
    "official_answer": ["A", "D"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 32,
    "subject": "Engineering Mathematics",
    "topic": "Differential Equations",
    "question_text": "The function y(t) satisfies t^2 y''(t) - 2ty'(t) + 2y(t)",
    "options": {},
    "has_circuit_or_graph": False,
    "detailed_solution": "This is a Cauchy-Euler differential equation. Assume a solution y(t) = t^r. Substituting yields the characteristic equation r(r-1) - 2r + 2 = 0, which simplifies to r^2 - 3r + 2 = 0. The roots are r=1 and r=2. The general solution is y(t) = C_1 t + C_2 t^2. The derivative is y'(t) = C_1 + 2C_2 t. Applying initial conditions: y'(0) = 1 means C_1 = 1. y'(1) = -1 means C_1 + 2C_2 = -1. Substituting C_1 gives 1 + 2C_2 = -1, so C_2 = -1. The specific solution is y(t) = t - t^2. To find the maximum on [0,1], set y'(t) = 1 - 2t = 0, yielding t = 0.5. The maximum value is y(0.5) = 0.5 - (0.5)^2 = 0.25.",
    "predicted_answer": ["0.25"],
    "official_answer": ["0.25"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 33,
    "subject": "Communications",
    "topic": "Information Theory and Coding",
    "question_text": "The generator matrix of a (6,3) binary linear block code",
    "options": {},
    "has_circuit_or_graph": False,
    "detailed_solution": "For a binary linear block code, the minimum Hamming distance d_min is equal to the minimum non-zero weight w_min of its codewords. The codewords are generated by combinations of the rows of the generator matrix G. The weights of the three individual rows are 3, 3, and 3. The linear combinations of any two rows (r1\\oplus r2, r1\\oplus r3, r2\\oplus r3) yield weights of 4, 4, and 4. The combination of all three rows (r1\\oplus r2\\oplus r3) yields [1 1 1 0 0 0], which has a weight of 3. The minimum non-zero weight among all possible combinations is 3, making d_min = 3.",
    "predicted_answer": ["3"],
    "official_answer": ["3"],
    "is_correct": True,
    "year": 2025,
    "image_path": None
  },
  {
    "question_number": 34,
    "subject": "Analog Circuits",
    "topic": "Active Filters",
    "question_text": "All the components in the bandpass filter given below are ideal.",
    "options": {},
    "has_circuit_or_graph": True,
    "detailed_solution": "This is an active inverting bandpass filter. The transfer function is H(s) = -Z_f / Z_i. The input impedance Z_i = R + 1/(s(10C)). The feedback impedance Z_f = 2R || 1/(s(0.1C)) = 2R / (1 + s(0.2RC)). The overall transfer function simplifies to H(s) = - (20sRC) / ((1 + 0.2sRC)(1 + 10sRC)). The poles define the cut-off frequencies. Pole 1 (lower freq) is at \\omega_1 = 1/(10RC). Pole 2 (upper freq) is at \\omega_2 = 1/(0.2RC) = 5/(RC). We are given that the lower -3dB frequency f_L is 1 MHz, which corresponds to \\omega_1. The upper frequency \\omega_2 = 5/(RC) can be rewritten as \\omega_2 = 50 * (1/(10RC)) = 50 * \\omega_1. Therefore, f_H = 50 * f_L = 50 * 1 MHz = 50 MHz.",
    "predicted_answer": ["50"],
    "official_answer": ["50"],
    "is_correct": True,
    "year": 2025,
    "image_path": "./images/Que_2025_Q34.jpg"
  },
  {
    "question_number": 35,
    "subject": "Digital Circuits",
    "topic": "Data Converters",
    "question_text": "A 4-bit weighted-resistor DAC with inputs b3, b2, b1, and b0",
    "options": {},
    "has_circuit_or_graph": True,
    "detailed_solution": "The circuit is an inverting summing amplifier acting as a DAC. The output voltage is V_O = - I_sum * R_feedback. The feedback resistor is R. The sum current is I_sum = V_REF * (b_3/R + b_2/2R + b_1/4R + b_0/8R). Therefore, V_O = - V_REF * (b_3 + b_2/2 + b_1/4 + b_0/8). Given V_REF = 2 V. For input 1110 (binary for 14), V_O1 = -2 * (1 + 0.5 + 0.25 + 0) = -3.5 V. For input 1101 (binary for 13), V_O2 = -2 * (1 + 0.5 + 0 + 0.125) = -3.25 V. The magnitude of the change in output voltage is |V_O2 - V_O1| = |-3.25 - (-3.5)| = 0.25 V, which is 250 mV.",
    "predicted_answer": ["250"],
    "official_answer": ["250"],
    "is_correct": True,
    "year": 2025,
    "image_path": "./images/Que_2025_Q35.jpg"
  }

]

with open("gate_database.json", "w") as f:
    json.dump(db_data, f)

# 2. Convert JSON into LangChain Documents
print("Loading Data into Documents...")
# Create a flat list just in case there are nested lists in the JSON
documents = []

for item in db_data:
    # 🛠️ MAGIC FIX: If the item is a list, loop through its contents!
    if isinstance(item, list):
        items_to_process = item
    else:
        items_to_process = [item]
        
    for sub_item in items_to_process:
        # Skip if it's missing vital keys
        if not isinstance(sub_item, dict) or 'subject' not in sub_item:
            continue
            
        # Combine topic and question text for highly accurate searching
        text_to_search = str(sub_item.get('topic', '')) + " " + str(sub_item.get('question_text', ''))
        
        doc = Document(
            page_content=text_to_search,
            metadata={
                "year": sub_item.get('year', 'Unknown'),
                "topic": sub_item.get('topic', 'Unknown'),
                "subject": sub_item.get('subject', 'Unknown'),
                "question_text": sub_item.get('question_text', ''),
                "detailed_solution": sub_item.get('detailed_solution', '')
            }
        )
        documents.append(doc)

print(f"Building FAISS Database with {len(documents)} questions...")
# 3. Build and Save FAISS Vector Database
print("Downloading Embedding Model (Takes ~10 seconds)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Building FAISS Vector Database...")
vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local("gate_faiss_index")
print("✅ Knowledge Base Built! Ready to run the App.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 61.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 require

/tmp/ipykernel_57/106265743.py:432: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building FAISS Vector Database...
✅ Knowledge Base Built! Ready to run the App.


In [4]:
!pip install schemdraw

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.4/148.4 kB 3.9 MB/s eta 0:00:00a 0:00:01


In [7]:
import os

# Create the .streamlit directory
os.makedirs('.streamlit', exist_ok=True)

# Write the config.toml file inside that directory
config_content = """
[theme]
base = "dark"
primaryColor = "#00ffcc"
backgroundColor = "#0e1117"
secondaryBackgroundColor = "#161a20"
textColor = "#e0e0e0"
"""

with open('.streamlit/config.toml', 'w') as f:
    f.write(config_content)

print("Configuration file created successfully!")

Configuration file created successfully!


In [13]:
%%writefile app.py
import streamlit as st
import ollama
import os
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, freqz, TransferFunction, step, impulse
import cv2 
import sympy as sp
import fitz  # PyMuPDF
from PIL import Image
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import schemdraw
import schemdraw.elements as elm


import base64

def get_base64_of_bin_file(bin_file):
    with open(bin_file, 'rb') as f:
        data = f.read()
    return base64.b64encode(data).decode()

# Convert your logo once at startup
try:
    img_base64 = get_base64_of_bin_file("/kaggle/input/datasets/mounikakalamraju/watermark/image (9).png")
except:
    img_base64 = None
    

# --- CUSTOM DARK MODE THEME ---
if img_base64:
    st.markdown(f"""
        <style>
        /* 1. Apply watermark to the main container area */
        [data-testid="stAppViewContainer"] {{
            background-image: url("data:image/png;base64,{img_base64}");
            background-repeat: no-repeat;
            background-position: center center;
            background-size: 100%; /* Adjust this to make the logo larger or smaller */
            background-attachment: fixed;
        }}

        /* 2. Dim the logo so it's a watermark */
        [data-testid="stAppViewContainer"]::before {{
            content: "";
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background-image: url("data:image/png;base64,{img_base64}");
            background-repeat: no-repeat;
            background-position: center;
            background-size: 100%;
            filter: brightness(0.1) opacity(0.9); /* This controls the watermark look */
            pointer-events: none; /* Allows user to click through the watermark */
            z-index: 0;
        }}

        /* 3. Ensure your chat input and main content float ABOVE the watermark */
        [data-testid="stMainBlockContainer"] {{
            position: relative;
            z-index: 1;
        }}
        </style>
    """, unsafe_allow_html=True)
    
# --- PAGE CONFIGURATION & CUSTOM CSS ---

st.set_page_config(page_title="Oasis Educational Platform", page_icon="🎓", layout="wide")



# --- LOAD FAISS RAG DB (FOR TUTOR MODE) ---
@st.cache_resource
def load_rag_db():
    try:
        embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        return FAISS.load_local("gate_faiss_index", embeddings, allow_dangerous_deserialization=True)
    except:
        return None
vectorstore = load_rag_db()

class CircuitSolver:
    @staticmethod
    def nodal_analysis(netlist):
        """
        Input format: [{'type': 'resistor', 'val': 100, 'nodes': ('A', 'B')}, ...]
        Returns: Symbolic solution dictionary
        """
        nodes = set()
        for comp in netlist: nodes.update(comp['nodes'])
        nodes = sorted([n for n in nodes if n != '0'])
        
        voltages = {n: sp.Symbol(f'V_{n}') for n in nodes}
        voltages['0'] = 0
        
        equations = []
        for node in nodes:
            kcl = 0
            for c in netlist:
                if node in c['nodes']:
                    n1, n2 = c['nodes']
                    n_other = n2 if n1 == node else n1
                    if c['type'] == 'resistor':
                        kcl += (voltages[node] - voltages.get(n_other, 0)) / c['val']
            equations.append(sp.Eq(kcl, 0))
        return sp.solve(equations, list(voltages.values()))

    @staticmethod
    def draw_circuit(netlist):
        """Generates a schematic plot."""
        d = schemdraw.Drawing()
        for c in netlist:
            if c['type'] == 'resistor':
                d += elm.Resistor().label(f"{c['val']}Ω")
            elif c['type'] == 'voltage_source':
                d += elm.SourceV().label(f"{c['val']}V")
        return d.draw()
        
# --- INITIALIZE STATE ---
# DSP Sandbox States
if "sandbox" not in st.session_state: st.session_state.sandbox = {"np": np, "pd": pd, "plt": plt, "cv2": cv2, "sp": sp,"CircuitSolver": CircuitSolver, "butter": butter, "filtfilt": filtfilt}
if "messages_ws" not in st.session_state: st.session_state.messages_ws = []
if "params" not in st.session_state: st.session_state.params = []
if "review" not in st.session_state: st.session_state.review = ""
if "last_code" not in st.session_state: st.session_state.last_code = ""
if "exports" not in st.session_state: st.session_state.exports = []
if "can_export_firmware" not in st.session_state: st.session_state.can_export_firmware = False
if "trigger_prompt" not in st.session_state: st.session_state.trigger_prompt = None
if "current_file" not in st.session_state: st.session_state.current_file = None

# Tutor States
if "messages_tutor" not in st.session_state: st.session_state.messages_tutor = []
if "final_answer" not in st.session_state: st.session_state.final_answer = None
if "rag_results" not in st.session_state: st.session_state.rag_results = []
if "is_exact_match" not in st.session_state: st.session_state.is_exact_match = False
if "detected_topic" not in st.session_state: st.session_state.detected_topic = ""

# --- SIDEBAR & FILE HANDLING ---
with st.sidebar:
    st.markdown("### 🔀 Select Platform Mode")
    app_mode = st.radio("Mode:", ["🛠️ Workspace", "🎓 GATE Tutor"])
    
    st.markdown("### ⚙️ AI Engine")
    model_choice = st.selectbox("Select Inference Model:", ["gemma4:e4b (Fast)", "gemma4:26b (Accurate)"])
    active_model = "gemma4:e4b" if "e4b" in model_choice else "gemma4:26b"
    
    st.markdown("---")
    uploaded_file = st.file_uploader("Upload CSV/Document/Image", type=["csv", "png", "jpg", "pdf"])
    
    if uploaded_file:
        save_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
        save_path = os.path.join(save_dir, uploaded_file.name)
        with open(save_path, "wb") as f: f.write(uploaded_file.getbuffer())
        
        # Optimize PDF / Image
        try:
            if save_path.lower().endswith('.pdf'):
                doc = fitz.open(save_path)
                page = doc.load_page(0) 
                pix = page.get_pixmap(matrix=fitz.Matrix(1, 1)) 
                img_path = save_path.replace('.pdf', '.png')
                pix.save(img_path)
                st.session_state["file_path"] = img_path 
                st.success("PDF Loaded and Optimized!")
                
            elif save_path.lower().endswith(('.png', '.jpg', '.jpeg')):
                img = Image.open(save_path)
                img.thumbnail((800, 800)) 
                img.save(save_path)
                st.session_state["file_path"] = save_path
                st.success("Image Loaded and Optimized!")
                
            else:
                st.session_state["file_path"] = save_path
                st.success("CSV Data Loaded!")
                
                # --- INSTANT NATIVE PREVIEW (Zero AI wait time!) ---
                df_prev = pd.read_csv(save_path)
                st.write(f"📊 **Rows:** {len(df_prev)} | **Cols:** {len(df_prev.columns)}")
                st.dataframe(df_prev.head(3))
                
        except Exception as e:
            st.error(f"Error processing file: {e}")

        # --- REMOVED THE AUTO-TRIGGER! The AI will now only think when YOU type a prompt. ---
        
    if "file_path" in st.session_state and st.sidebar.button("Remove", use_container_width=True):
        del st.session_state["file_path"]
        st.session_state.current_file = None
        st.rerun()

#st.markdown("<h1>Oasis Educational Platform</h1>", unsafe_allow_html=True)
#st.markdown("<h1 style='text-align: center; margin-top: 0px;'>Oasis Educational Platform</h1>", unsafe_allow_html=True)
#col_main, col_right = st.columns([7.5, 2.5])

# ==========================================
# MODE 1: 🛠️ WORKSPACE (Interactive DSP Lab)
# ==========================================
if app_mode == "🛠️ Workspace":
    
    sys_prompt_dsp = """You are an Advanced Data Analysis and DSP Engine. Variables persist in memory.
    RULES:
    1. Output ONLY Python code in ```python ``` blocks. Keep code fast and minimal.
    2. PLOTTING: Always start with plt.clf(). End with plt.savefig('output.png', bbox_inches='tight').
    3. PARAMETERS: Deduce tuning parameters. Format: [PARAM: Name | Current | Opt1, Opt2, Custom]
    4. REVIEW: Output [REVIEW: Your 1-sentence critique].
    5. EXPORTS: If data is processed, save it (e.g., df.to_csv('processed.csv')) and output [EXPORT: processed.csv].
    6. FIRMWARE: Output [FIRMWARE: YES] if math can be exported to C coefficients. Otherwise [FIRMWARE: NO].
    """
    if not st.session_state.messages_ws: st.session_state.messages_ws = [{"role": "system", "content": sys_prompt_dsp}]

 #   st.markdown("<h1>🛠️ Interactive DSP Workspace</h1>", unsafe_allow_html=True)
    col_main, col_right = st.columns([7.5, 2.5])
    user_input = st.chat_input("E.g., Apply a 50Hz notch filter to the data...")
    active_prompt = st.session_state.trigger_prompt or user_input

    # --- MAIN DSP SCREEN ---
    with col_main:
        status_placeholder = st.empty()
        plot_placeholder = st.empty()
        
        if os.path.exists("output.png") and not active_prompt:
            plot_placeholder.image("output.png", use_container_width=True)
            
        if st.session_state.params and not active_prompt:
            table_html = "<table class='specs-table'><tr><th>Parameter</th><th>Current Value</th></tr>"
            for p in st.session_state.params:
                table_html += f"<tr><td>{p['name']}</td><td><b>{p['current']}</b></td></tr>"
            table_html += "</table>"
            st.markdown(table_html, unsafe_allow_html=True)
            
        if st.session_state.review and not active_prompt:
            st.info(f"💡 **AI Profiling:** {st.session_state.review}")
            
        if st.session_state.last_code and not active_prompt:
            with st.expander("🧑‍💻 View & Edit Python Sandbox Code"):
                edited_code = st.text_area("Modify and execute sandbox code:", value=st.session_state.last_code, height=200)
                if st.button("▶️ Run Edited Code", type="primary"):
                    try:
                        if os.path.exists("output.png"): os.remove("output.png")
                        exec(edited_code, st.session_state.sandbox)
                        st.session_state.last_code = edited_code
                        st.success("Execution applied!")
                        time.sleep(1)
                        st.rerun()
                    except Exception as e:
                        st.error(f"Manual Error: {e}")

    # --- DSP EXECUTION ENGINE ---
    if active_prompt:
        st.session_state.trigger_prompt = None
        plot_placeholder.empty()
        st.session_state.messages_ws.append({"role": "user", "content": active_prompt})
        
        with col_main:
            start_time = time.time()
            full_response = ""
            with st.spinner("🧠 AI Engine Computing..."):
                for chunk in ollama.chat(model=active_model, messages=st.session_state.messages_ws, stream=True):
                    full_response += chunk['message']['content']
            
            st.session_state.messages_ws.append({"role": "assistant", "content": full_response})
            
            # Extract Tags
            param_matches = re.findall(r"\[?PARAM:\s*(.*?)\s*\|\s*(.*?)\s*\|\s*(.*?)\]?", full_response, re.IGNORECASE)
            if param_matches:
                parsed_params = []
                seen_names = set() 
                for p in param_matches:
                    name = p[0].strip()
                    if name in seen_names: continue
                    seen_names.add(name)
                    opts = [o.strip() for o in p[2].split(",")]
                    curr = p[1].strip()
                    if "Custom" not in opts: opts.append("Custom")
                    if curr not in opts: opts.insert(0, curr)
                    parsed_params.append({"name": name, "current": curr, "options": opts})
                st.session_state.params = parsed_params
                
            review_match = re.search(r"\[?REVIEW:\s*(.*?)\]?", full_response, re.IGNORECASE)
            if review_match: st.session_state.review = review_match.group(1).strip()
                
            fw_match = re.search(r"\[?FIRMWARE:\s*(YES|NO)\]?", full_response, re.IGNORECASE)
            if fw_match: st.session_state.can_export_firmware = (fw_match.group(1).upper() == "YES")

            export_matches = re.findall(r"\[?EXPORT:\s*(.*?)\]?", full_response, re.IGNORECASE)
            st.session_state.exports = [m.strip() for m in export_matches]

            code_match = re.search(r"```python(.*?)```", full_response, re.DOTALL | re.IGNORECASE)
            if code_match:
                python_code = code_match.group(1).strip()
                st.session_state.last_code = python_code 
                status_placeholder.warning("⚙️ Executing in Stateful Sandbox...")
                try:
                    if os.path.exists("output.png"): os.remove("output.png")
                    exec(python_code, st.session_state.sandbox)
                    if os.path.exists("output.png"):
                        status_placeholder.success(f"✅ Completed in {time.time() - start_time:.2f}s.")
                    time.sleep(0.8)
                    status_placeholder.empty()
                except Exception as e:
                    status_placeholder.error(f"⚠️ Error: {e}")
            st.rerun()

    # --- DSP CONTROL PANEL (RIGHT) ---
    if st.session_state.params and not active_prompt:
        with col_right:
            st.markdown("### 🎛️ Tune Settings")
            new_values = {}
            for i, p in enumerate(st.session_state.params):
                selected = st.selectbox(p['name'], options=p['options'], index=p['options'].index(p['current']), key=f"sel_{i}_{p['name']}")
                if selected == "Custom":
                    custom_val = st.text_input(f"Enter custom {p['name']}:", key=f"txt_{i}_{p['name']}")
                    new_values[p['name']] = custom_val if custom_val else selected
                else:
                    new_values[p['name']] = selected
                    
            if st.button("🔄 Update Model", use_container_width=True, type="primary"):
                changes = [f"Change {p['name']} to {new_values[p['name']]}" for p in st.session_state.params if new_values.get(p['name']) and new_values[p['name']] != "Custom" and new_values[p['name']] != p['current']]
                if changes:
                    st.session_state.trigger_prompt = "Update: " + " and ".join(changes) + ". Use memory. Output [PARAM], [REVIEW], [EXPORT], [FIRMWARE]."
                    st.rerun()
                    
            st.markdown("---")
            st.markdown("### 💾 Workspace Exports")
            for export_file in st.session_state.exports:
                if os.path.exists(export_file):
                    with open(export_file, "rb") as f:
                        file_ext = export_file.split('.')[-1].upper()
                        st.download_button(f"📥 Download {file_ext}", data=f, file_name=export_file, use_container_width=True)
                        
            if st.session_state.can_export_firmware:
                if st.button("🚀 Gen .c Header", use_container_width=True):
                    st.session_state.trigger_prompt = "Write filter coefficients to 'firmware.h'."
                    st.rerun()
                if os.path.exists("firmware.h"):
                    with open("firmware.h", "rb") as f:
                        st.download_button("📥 Download firmware.h", data=f, file_name="firmware.h", type="primary", use_container_width=True)


# ==========================================
# MODE 2: 🎓 GATE TUTOR (Autonomous VLM + RAG)
# ==========================================
elif app_mode == "🎓 GATE Tutor (Autonomous AI)":
    st.markdown("<h1>🎓 GATE ECE Educator</h1>", unsafe_allow_html=True)
    
    sys_prompt_tutor = """You are an expert GATE ECE Tutor.
    Read the user's image. You MUST format your response EXACTLY in this order:
    [TOPIC: Write 1-2 words identifying the core subject, e.g., BJT Biasing, Nyquist Plot]
    [TRANSCRIBED: Write out the actual text/equation of the question you see in the image]
    [REASONING: Write your step-by-step mathematical reasoning in Markdown/LaTeX]
    [ANSWER: Option X or final value]
    """
    if not st.session_state.messages_tutor: st.session_state.messages_tutor = [{"role": "system", "content": sys_prompt_tutor}]

    col_left, col_right = st.columns([4, 6])
    
    with col_left:
        st.markdown("### 📄 Uploaded Question")
        if "file_path" in st.session_state and st.session_state["file_path"].endswith(('.png', '.jpg')):
            st.image(st.session_state["file_path"], use_container_width=True)
        else:
            st.info("👈 Upload a PDF or screenshot of a GATE question.")
            
        st.markdown("---")
        st.markdown("### 📊 Autonomous Analytics")
        if st.session_state.rag_results:
            if st.session_state.is_exact_match:
                st.markdown("<div class='exact-match'>🚨 PREVIOUS YEAR QUESTION DETECTED!</div>", unsafe_allow_html=True)
            top_match = st.session_state.rag_results[0][0].metadata 
            st.write(f"**AI Detected Topic:** {st.session_state.detected_topic}")
            st.write(f"**Subject:** {top_match.get('subject', 'N/A')}")
            if st.session_state.is_exact_match:
                st.write(f"**Originally Asked By:** {top_match.get('iit_institution', 'Unknown')} (GATE {top_match.get('year', 'Unknown')})")
            
            st.markdown("---")
            st.markdown("**📚 Related Past Questions:**")
            for doc, score in st.session_state.rag_results:
                meta = doc.metadata
                with st.expander(f"GATE {meta.get('year', 'N/A')} - {meta.get('topic', 'N/A')}"):
                    st.markdown(f"**Question:** {meta.get('question_text', 'N/A')}")
                    st.markdown(f"**Solution:** {meta.get('detailed_solution', 'N/A')}")
        else:
            st.caption("Analytics will appear automatically after the AI analyzes the question.")

    with col_right:
        st.markdown(f"### 🧠 AI Reasoning Engine ({active_model})")
        for msg in st.session_state.messages_tutor:
            if msg["role"] == "assistant":
                clean_text = re.sub(r"\[?TOPIC:\s*(.*?)\]?", "", msg["content"], flags=re.IGNORECASE)
                clean_text = re.sub(r"\[?TRANSCRIBED:\s*(.*?)\]?", "", clean_text, flags=re.IGNORECASE)
                clean_text = re.sub(r"\[?REASONING:\s*", "", clean_text, flags=re.IGNORECASE)
                clean_text = re.sub(r"\[?ANSWER:\s*(.*?)\]?", "", clean_text, flags=re.IGNORECASE)
                st.markdown(clean_text.strip())
                
        if st.session_state.final_answer:
            st.markdown(f"<div class='big-answer'>✅ Final Answer: {st.session_state.final_answer}</div>", unsafe_allow_html=True)

    user_input = st.chat_input("Click to solve...", key="tutor_input")

    if user_input:
        user_msg = {"role": "user", "content": user_input}
        if "file_path" in st.session_state and st.session_state["file_path"].endswith(('.png', '.jpg')):
            user_msg["images"] = [st.session_state["file_path"]]
            
        st.session_state.messages_tutor.append(user_msg)
        
        with col_right:
            status = st.empty()
            start_time = time.time()
            full_response = ""
            
            with st.spinner("👁️ AI is digesting the image and calculating..."):
                for chunk in ollama.chat(model=active_model, messages=st.session_state.messages_tutor, stream=True):
                    full_response += chunk['message']['content']
                    elapsed = time.time() - start_time
                    display_text = re.sub(r"\[.*?\]", "", full_response) 
                    status.markdown(f"**⏱️ {elapsed:.1f}s** \n\n" + display_text + "▌")
            status.empty()
            
            topic_match = re.search(r"\[?TOPIC:\s*(.*?)\]?", full_response, re.IGNORECASE)
            st.session_state.detected_topic = topic_match.group(1).strip() if topic_match else "Unknown Topic"
            
            transcribed_match = re.search(r"\[?TRANSCRIBED:\s*(.*?)\]?", full_response, re.IGNORECASE)
            search_text = transcribed_match.group(1).strip() if transcribed_match else st.session_state.detected_topic
            
            ans_match = re.search(r"\[?ANSWER:\s*(.*?)\]?", full_response, re.IGNORECASE)
            st.session_state.final_answer = ans_match.group(1).strip() if ans_match else None
            
            if vectorstore and search_text:
                results = vectorstore.similarity_search_with_score(search_text, k=3)
                st.session_state.rag_results = results
                st.session_state.is_exact_match = False
                if results and results[0][1] < 0.6: 
                    st.session_state.is_exact_match = True
                
            st.session_state.messages_tutor.append({"role": "assistant", "content": full_response})
            st.rerun()

Overwriting app.py


In [14]:
import subprocess
import time
import os
import urllib.request
import re

print("1. Starting Streamlit in the background...")
# Run Streamlit using Python's subprocess instead of bash
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("2. Downloading Cloudflare Tunnel...")
if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", 
        "cloudflared"
    )
    os.chmod("cloudflared", 0o755) # Make it executable

print("3. Starting Cloudflare Tunnel...")
# Open a log file and run cloudflared, redirecting output to the file
with open("tunnel.log", "w") as log_file:
    subprocess.Popen([
        "./cloudflared", "tunnel", "--url", "http://localhost:8501"
    ], stdout=log_file, stderr=log_file)

print("4. Waiting for connection... (takes about 5-8 seconds)")
time.sleep(8)

# Read the log file to find the URL
print("\n=========================================================")
with open("tunnel.log", "r") as log_file:
    log_content = log_file.read()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_content)
    
    if match:
        print("✅ YOUR APP IS READY!")
        print("Click the link below to open OASIS:")
        print(match.group(0))
    else:
        print("⚠️ Tunnel is still starting. Wait 5 seconds and run this cell again.")
print("=========================================================\n")

1. Starting Streamlit in the background...
2. Downloading Cloudflare Tunnel...
3. Starting Cloudflare Tunnel...
4. Waiting for connection... (takes about 5-8 seconds)

✅ YOUR APP IS READY!
Click the link below to open OASIS:
https://treo-putting-buddy-houston.trycloudflare.com

